# SCP Step 5 (Full Build, Inputs, Simulation, Analysis) - Colab Classroom Notebook

This is the end-to-end classroom notebook for the SCP single-cell pipeline.
It can run on fresh Colab runtimes and supports notebook-only usage.

## What this notebook covers
1. Environment/bootstrap setup (repo, dependencies, mechanisms).
2. Cell loading and geometry definition.
3. Input generation and synapse preview.
4. Simulation run modes (single or multi-style workflows).
5. Quick analysis plots for interpretation.

## Recommended first run
1. Run top-to-bottom without edits.
2. Confirm one successful baseline run.
3. Then edit one block at a time (cell choice, timing, amplitudes, input groups).

## What students should edit first
- `cell_name`, `model_dir` in the tune-selection cell.
- stimulation/input parameters in later step cells.
- plotting windows only after simulation runs successfully.


In [ ]:
# Colab/Linux bootstrap: locate or clone SCP, install deps, and set repo root
import os
import sys
import subprocess
import importlib
from pathlib import Path


def _is_scp_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "cells").is_dir()
        and (path / "modules").is_dir()
        and (path / "run_pipeline.py").is_file()
    )


def _find_scp_root(start: Path):
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if _is_scp_root(cand):
            return cand
    try:
        for child in start.iterdir():
            if child.is_dir() and _is_scp_root(child):
                return child.resolve()
    except Exception:
        pass
    return None


def _repo_url_with_token(repo_url: str) -> str:
    token = (
        os.environ.get("SCP_GIT_TOKEN")
        or os.environ.get("SCP_GITHUB_TOKEN")
        or os.environ.get("GITHUB_TOKEN")
    )
    if token and repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@", 1)
    return repo_url


def _redact_repo_url(url: str) -> str:
    if "://" not in url or "@" not in url:
        return url
    scheme, rest = url.split("://", 1)
    host_part = rest.split("@", 1)[1]
    return f"{scheme}://***@{host_part}"


def _clone_repo(repo_url: str, repo_dir: Path, branch=None) -> Path:
    repo_dir = repo_dir.expanduser().resolve()
    if _is_scp_root(repo_dir):
        return repo_dir

    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise FileExistsError(
            f"Repo dir exists but is not an SCP checkout: {repo_dir}. "
            "Set SCP_REPO_DIR to an empty folder or existing SCP clone."
        )

    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    auth_url = _repo_url_with_token(repo_url)

    cmd = ["git", "clone", "--depth", "1"]
    if branch:
        cmd += ["--branch", branch]
    cmd += [auth_url, str(repo_dir)]

    shown_cmd = [*cmd[:-2], _redact_repo_url(auth_url), str(repo_dir)]
    print("Cloning SCP repo:", " ".join(shown_cmd))

    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Failed to clone SCP repo. If this repo is private, set SCP_GIT_TOKEN (or SCP_GITHUB_TOKEN/GITHUB_TOKEN) in Colab secrets/environment."
        ) from exc

    if not _is_scp_root(repo_dir):
        raise RuntimeError(f"Clone completed but repo layout is invalid: {repo_dir}")
    return repo_dir


def _ensure_python_pkg(import_name: str, pip_name=None) -> None:
    try:
        importlib.import_module(import_name)
        return
    except Exception:
        pass
    pkg = pip_name or import_name
    print(f"Installing missing package: {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
AUTO_CLONE_REPO = os.environ.get("SCP_AUTO_CLONE", "1") not in ("0", "false", "False")
DEFAULT_REPO_URL = "https://github.com/HunterBushnell/SCP.git"
REPO_URL = os.environ.get("SCP_REPO_URL", DEFAULT_REPO_URL)
REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
DEFAULT_REPO_DIR = Path("/content/SCP") if IN_COLAB else (Path.cwd() / "SCP")
REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", str(DEFAULT_REPO_DIR)))

repo_root = _find_scp_root(Path.cwd())
cloned = False
if repo_root is None and AUTO_CLONE_REPO:
    repo_root = _clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
    cloned = True

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate SCP repo from current directory. "
        "Set SCP_REPO_DIR/SCP_REPO_URL or run notebook from inside SCP."
    )

repo_root = repo_root.resolve()
os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("SCP root:", repo_root)
from modules.notebook_helpers import ensure_scp_repo_on_syspath
repo_root = ensure_scp_repo_on_syspath(repo_root)
os.environ["SCP_ROOT"] = str(repo_root)
if cloned:
    print("SCP was cloned for this session.")

# Only needed on fresh Colab sessions
INSTALL_DEPS = IN_COLAB
if INSTALL_DEPS:
    _ensure_python_pkg("numpy")
    _ensure_python_pkg("pandas")
    _ensure_python_pkg("matplotlib")
    _ensure_python_pkg("scipy")
    _ensure_python_pkg("h5py")
    _ensure_python_pkg("neuron")
    # Required by modules.load_cell used in Step 5
    _ensure_python_pkg("allensdk")

    # Needed to compile NEURON modfiles when x86_64 binaries are absent
    subprocess.check_call(["apt-get", "update"])
    subprocess.check_call(["apt-get", "install", "-y", "build-essential"])

# Required external inputs (print if missing)
required_external = [
    repo_root / "external_data" / "pyrFiringRateAvg.csv",
    repo_root / "external_data" / "PV_1000tr.pkl",
    repo_root / "external_data" / "SST_1000tr.pkl",
]
missing = [p for p in required_external if not p.is_file()]
if missing:
    print("Missing external inputs:")
    for p in missing:
        print(" -", p)


def ensure_modfiles(tune_dir: Path, compile_modfiles: bool = True) -> None:
    mod_dir = tune_dir / "modfiles"
    dll_candidates = [
        mod_dir / "x86_64" / ".libs" / "libnrnmech.so",
        mod_dir / "x86_64" / "libnrnmech.so",
    ]
    if any(p.is_file() for p in dll_candidates):
        return
    if not mod_dir.is_dir():
        print(f"Missing modfiles dir: {mod_dir}")
        return
    if compile_modfiles:
        print("Compiling modfiles with nrnivmodl...")
        subprocess.check_call(["nrnivmodl"], cwd=str(mod_dir))
    else:
        print("Missing compiled modfiles; run nrnivmodl in", mod_dir)


## Quick Start and Instructor Notes

- If users only have this notebook, the bootstrap cell can clone SCP automatically.
- For private repos, provide a token via `SCP_GIT_TOKEN` or `GITHUB_TOKEN`.
- Keep `cell_configs` files as the source of truth for class labs.

### Suggested class flow
1. Run setup and tune selection.
2. Build cell and geometry.
3. Generate inputs and preview synapses.
4. Run one short simulation.
5. Discuss outputs before parameter exploration.


In [ ]:
# Dev helper: autoreload modules and ensure repo on sys.path
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

try:
    from modules.notebook_helpers import ensure_scp_repo_on_syspath
except ModuleNotFoundError:
    env_root = os.environ.get("SCP_ROOT")
    injected = False
    if env_root:
        cand = Path(env_root).expanduser().resolve()
        if (cand / "modules").is_dir() and (cand / "run_pipeline.py").is_file():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            injected = True

    if not injected:
        start = Path.cwd().resolve()
        for cand in (start, *start.parents):
            if (cand / "modules").is_dir() and (cand / "run_pipeline.py").is_file():
                if str(cand) not in sys.path:
                    sys.path.insert(0, str(cand))
                injected = True
                break

    if not injected:
        raise ModuleNotFoundError("Could not import modules. Run the bootstrap cell first.")

    from modules.notebook_helpers import ensure_scp_repo_on_syspath

repo_root = ensure_scp_repo_on_syspath()
os.environ["SCP_ROOT"] = str(repo_root)


In [ ]:
import os, sys, csv, json, h5py, random, math, pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

from allensdk.api.queries.biophysical_api import BiophysicalApi
from allensdk.model.biophys_sim.config import Config
from allensdk.model.biophysical.utils import Utils


# from modules import download_cell

from modules import load_cell
from modules import geometry
from modules import inputs
from modules import synapses

from modules import run_sim
from modules.analysis import plotting


In [ ]:
# Select the cell/tune and move into the tune directory (selected tune context)
cell_name = 'PV'  # SST, PV, PN
model_dir = 'seg_tuned'

tune_dir = (repo_root / 'cells' / cell_name / 'tunes' / model_dir).resolve()
if not tune_dir.is_dir():
    raise FileNotFoundError(f'Expected tune dir not found: {tune_dir}')
os.chdir(tune_dir)
print('CWD:', Path.cwd())

# Defaults from cell_configs (fallback if sim_config.json is missing)
cell_cfg_defaults = {}
cell_cfg_path = Path('cell_configs') / 'cell_config.json'
if cell_cfg_path.is_file():
    cell_cfg_defaults = json.loads(cell_cfg_path.read_text())



# Override from sim_config.json when available (cell_configs preferred)
sim_cfg_path = Path('cell_configs') / 'sim_config.json'
if not sim_cfg_path.is_file():
    sim_cfg_path = Path('sim_config.json')
sim_cfg_preview = None
if sim_cfg_path.is_file():
    sim_cfg_preview = json.loads(sim_cfg_path.read_text())


In [ ]:
# Ensure compiled modfiles exist (prints if missing)
ensure_modfiles(tune_dir, compile_modfiles=IN_COLAB)


In [ ]:
from pathlib import Path
from neuron import h

h.load_file("stdrun.hoc")  # Required to use h.run()

# Support both common nrnivmodl outputs: x86_64/.libs/libnrnmech.so and x86_64/libnrnmech.so
dll_candidates = [
    Path("modfiles/x86_64/.libs/libnrnmech.so"),
    Path("modfiles/x86_64/libnrnmech.so"),
]
dll_path = next((p for p in dll_candidates if p.is_file()), None)
if dll_path is None:
    raise FileNotFoundError(
        "Compiled mechanisms not found. Run nrnivmodl in modfiles/ first."
    )

h.nrn_load_dll(str(dll_path))
print(f"Loaded mechanisms from {dll_path}")


In [ ]:
# Load in-vivo firing-rate curve once (PN)
from modules.analysis import bio_curve

in_vivo_curve_raw = bio_curve.load_bio_curve(
    str(repo_root / "external_data" / "pyrFiringRateAvg.csv"),
    time_col="Time",
    rate_col="AvgFiringRate",
    t_min=0.0,
    delay_ms=0.0,
    time_unit="s",
)

# Default (unshifted); will be aligned after groups_cfg is available
in_vivo_curve = in_vivo_curve_raw


# Step 5.2 - Build Cell

This section constructs a single NEURON cell, ready for simulation, in four substeps:

5.2.1 **Load Cell**  
: Build the NEURON cell object from the tuned model files.

5.2.2 **Define Geometry**  
: Create a standardized geometry view with named segment groups (e.g. soma, proximal dendrites).

5.2.3 **Generate Inputs**  
: Generate spike trains for each synapse group (homogeneous, inhomogeneous, fixed, etc.), independent of morphology.

5.2.4 **Preview Synapses**  
: Preview synapse placement and weights without attaching NEURON objects.


## 5.2.1 Load Cell

In this step, we construct the NEURON cell object from the tuned model files.  
The notebook will pass a configuration dictionary (`cell_config`) to a single function `load_cell(...)` in `modules`, which internally handles all model-specific details (e.g., Allen vs custom, manifest paths, soma tuning).  
The result is a `cell` handle that owns the NEURON sections and is ready for further processing.


In [ ]:
# Minimal cell configuration for the current tuned Allen model
cell_config_path = Path("cell_configs") / "cell_config.json"
if cell_config_path.is_file():
    cell_config = json.loads(cell_config_path.read_text())
else:
    cell_config = {}

cell_config.setdefault("cell_name", cell_name)
paths = cell_config.setdefault("paths", {})
paths.setdefault("manifest", "manifest.json")

tuning = cell_config.get("tuning")
if not isinstance(tuning, dict) or "soma_diam_multiplier" not in tuning:
    raise KeyError("cell_configs/cell_config.json must define tuning.soma_diam_multiplier. Run Step 0 scaffold or set it manually before Step 5.")
tuning["soma_diam_multiplier"] = float(tuning["soma_diam_multiplier"])

# tune directory defines model identity; do not copy specimen/type into cell_config.


In [ ]:
from modules.load_cell import load_cell

cell = load_cell(cell_config)

print("Loaded cell:", cell)


## 5.2.1b Current Injection Test (optional)

Quick somatic IClamp to check base cell behavior without synapses/inputs.


In [ ]:
# IClamp mode (cell-only), controlled by cell_configs/sim_config.json
iclamp_cfg = (sim_cfg_preview or {}).get("iclamp", {}) if "sim_cfg_preview" in globals() else {}
run_iclamp_mode = bool(iclamp_cfg.get("enabled", False))
if run_iclamp_mode:
    print("IClamp mode enabled (cell_configs/sim_config.json -> iclamp.enabled = true)")


## 5.2.2 Define Geometry

Here we create a standardized geometry view of the cell that is independent of how the cell was originally built.  
Given the `cell` and a configuration dictionary (`geom_config`), the function `define_geometry(...)` in `modules` will group segments into named sets (e.g., `soma`, `prox_dend`, `dist_dend`, `all_dend`) and compute path distances from a defined origin.  
The result is a `geom` structure that maps group names to lists of target locations `(sec, x, dist_um)` for synapse placement.


In [ ]:
# Geometry configuration (loaded from tune-specific config)
geom_config_path = Path("cell_configs") / "geometry.json"
if geom_config_path.is_file():
    geom_config = json.loads(geom_config_path.read_text())
else:
    geom_config = {}

geom_config.setdefault("label", f"{cell_name}_default_geometry")


In [ ]:
from modules.geometry import define_geometry

geom = define_geometry(cell, geom_config)

print("Geometry label:", geom.get("label", "<unnamed>"))
print("Geometry groups:", list(geom.get("groups", {}).keys()))

## 5.2.3 Generate Inputs

In this step we generate spike trains for each synapse group, completely separate from the cell morphology.  
The function `generate_inputs(...)` in `modules` will use `input_config`, `sim_params`, and `syn_params` to create spike-time arrays from various source types (e.g., fixed trains, homogeneous Poisson, inhomogeneous bio traces, baseline+bio windows).  
The output `inputs` is a pure Python/NumPy structure (no NEURON objects) organized per population and synapse group.


In [ ]:
from pathlib import Path
from modules import inputs

# working dir = tune folder
sim_cfg_preview, groups_cfg_preview = inputs.check_inputs()
print("Simulation config preview:", sim_cfg_preview)
print("Groups config preview:", groups_cfg_preview)

In [ ]:
# === Step 2.3: Generate Inputs ===
if run_iclamp_mode:
    sim_cfg = inputs._normalize_sim_config(sim_cfg_preview)
    inputs._inject_path_metadata(sim_cfg, sim_cfg_path.parent)
    groups_cfg = {}
    inputs_by_group = {}
    print("IClamp mode: skipping inputs generation.")
else:
    sim_cfg, groups_cfg, inputs_by_group = inputs.generate_inputs(geometry=geom)

    # print(inputs_by_group)
    print("sim_cfg:", sim_cfg)
    print("save:", sim_cfg.get("save"), "append:", sim_cfg.get("append"))
    print("Generated input groups:")
    for name, gi in inputs_by_group.items():
        print(f"  - {name:15s} mode={gi.mode!r:18}  n_trains={len(gi.spike_trains)}")
        
    print("seed:", sim_cfg.get("seed"))
    print("rand_seed:", sim_cfg.get("random_seed"))
    print("rand_global_seed:", sim_cfg.get("randomness", {}).get("global", {}).get("seed"))


## 5.2.4 Preview Synapses

Preview synapse placement and weights without attaching NEURON objects.  
Use `synapses.preview_synapses(...)` to generate a `syn_state` record for plotting.  
Actual synapse attachment happens inside Step 5.3 (run_sim) so single and multi runs stay consistent.


In [ ]:
# === Step 2.4: Preview synapse placement (no NEURON objects) ===
if run_iclamp_mode:
    print("IClamp mode: skipping synapse preview.")
else:
    from modules import synapses, randomness

    # 2.3 outputs: sim_cfg, groups_cfg, inputs_by_group
    # 2.1/2.2: cell, geom

    preview_synapses = True
    if preview_synapses:
        rm_preview = randomness.RandomnessManager(sim_cfg)
        trial_rng = rm_preview.trial(0)
        syn_state = synapses.preview_synapses(
            cell=cell,
            geom=geom,
            sim_cfg=sim_cfg,
            groups_cfg=groups_cfg,
            inputs_by_group=inputs_by_group,
            trial_rng=trial_rng,
        )

        print("Step 2.4: synapse preview complete (no NEURON objects attached).")
        print("Synapse groups:", list(syn_state.get("records", {}).keys()))
        for gname, records in syn_state.get("records", {}).items():
            print(f"  {gname}: {len(records)} synapse records")
    else:
        print("Synapse preview disabled.")


In [ ]:
# --- Debug synapse records (safe) ---
if "syn_state" not in globals() or syn_state is None:
    print("syn_state not available (preview not run).")
else:
    for g, recs in syn_state.get("records", {}).items():
        if not recs:
            continue
        print(g, "n_syn:", len(recs), "example weight:", recs[0].weight, "first spikes:", recs[0].spike_times[:5])
    if "groups_cfg" in globals():
        print("Groups configuration:", groups_cfg)


# Step 5.3 - Run Simulations

Use this section for the core experimental run.

### Typical class progression
1. Run one baseline simulation.
2. Change one parameter set (for example input rate or current clamp).
3. Re-run and compare against baseline.

Keep notes on exactly what changed between runs.


In [ ]:
# === Step 3.1: Run simulation (IClamp or normal) ===
from modules import run_sim

if run_iclamp_mode:
    print("Running IClamp test (no synapses/inputs).")
    results = run_sim.run_iclamp_test(cell, sim_cfg, iclamp_cfg=iclamp_cfg)
else:
    # sim_cfg comes from Step 2.3 and already has dt, tstart, tstop, seed, etc.
    results = run_sim.run_sim(
        cell,
        geom,
        sim_cfg,
        groups_cfg,
        inputs_by_group,
        meta_overrides={"cell_config": cell_config, "geometry_config": geom_config},
    )
    run_sim.summarize_results(results)
# single-trial case (works for IClamp too)
T = results["traces"]["T"]
V = results["traces"]["V"]
spikes = results.get("spikes")


### Step 5.3.4 - Save Results (auto)

Saves using the same workflow as the CLI/slurm pipeline.


In [ ]:
# --- Manual save (optional; auto-save already happens inside run_sim.run_sim) ---
do_manual_save = False
if do_manual_save:
    from datetime import datetime
    from pathlib import Path
    
    save_output = sim_cfg.get("save_output", True)
    if save_output is None:
        save_output = True
    save_output = bool(save_output)
    
    if save_output:
        if not sim_cfg.get("output"):
            sim_cfg["output"] = datetime.now().strftime("nb_%Y%m%d_%H%M%S")
        out_dir = Path("output_data")
        saved_path = run_sim.save_results(results, base_dir=out_dir)
        if saved_path is None:
            print("Results not saved (sim_cfg['output'] was empty).")
        else:
            print(f"Results saved to {saved_path}")
    else:
        print("Auto-save disabled (sim_cfg['save_output']=False).")
else:
    print("Skipping manual save (auto-save handled by run_sim.run_sim).")


In [ ]:
# --- Quick stats: spikes per trial ---
if run_iclamp_mode:
    print("Skipping spike stats (IClamp mode).")
else:
    from modules.analysis import summarize_spike_trials

    plot_trial_stats = True
    trial_stats = summarize_spike_trials(results, plot=plot_trial_stats)
    # trial_stats


# Step 5.4 - Analyze Results

Use these plots to answer:
- Did the cell respond in the expected time window?
- Are firing rates and trends plausible?
- Do synapse/input summaries match configured intent?

For class reports, capture:
1. one voltage-trace figure,
2. one spike/rate summary,
3. a short explanation of parameter changes and effects.


### Step 5.4.1 Cell & Synapse Generation


In [ ]:
if run_iclamp_mode:
    print("Skipping Step 4.1 (IClamp mode).")
else:
    # === Step 4.1: Analyze Cell & Synapse Generation ===
    # Align in-vivo curve to PN stimulation timing (if available)
    stim_group = "pn_exc"
    try:
        timing = groups_cfg.get(stim_group, {}).get("timing", {})
        stim_tstart = timing.get("stim_tstart_ms")
        input_stim = timing.get("input_stim_tstart_ms")
        if stim_tstart is None:
            stim_tstart = sim_cfg.get("stim_start_ms")
        if stim_tstart is not None:
            shift_ms = float(stim_tstart)
            if input_stim is not None:
                shift_ms = float(stim_tstart) - float(input_stim)
            t_s, rate = in_vivo_curve_raw
            in_vivo_curve = (t_s + shift_ms / 1000.0, rate)
    except Exception:
        pass


In [ ]:
# === Step 4.3: Analyze Single Sim ===
from modules.analysis import plotting

if run_iclamp_mode:
    plt.figure(figsize=(6, 4))
    plt.plot(T, V, lw=1.5)
    plt.xlabel("Time (ms)")
    plt.ylabel("Vm (mV)")
    plt.title("IClamp test (soma)")
    plt.tight_layout()
else:
    plotting.plot_results(
        results,
        syn_records=results.get("syn_records"),
        in_vivo_curve=in_vivo_curve,
        win_size=25,
        raster_style='dot',
        plot_window=(None, None),  # or (None, None) to auto
    )


# Extra Modules (optional)


In [ ]:
sys.exit("Stopping notebook here")

In [ ]:
# Check the input synapse group params
# to check timing and rates look correct
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from modules import input_sampling

tune_dir = (repo_root / 'cells' / cell_name / 'tunes' / model_dir).resolve()
centers, mean_rate, std_rate, sim_cfg, meta, ref_curve = input_sampling.sample_group_rates(
    tune_dir=tune_dir,
    group="sst_inh",
    runs=2000,
    bin_ms=None,
    seed=None,
)


# Save or inspect
df = pd.DataFrame({
    "time_ms": centers,
    "rate_mean_hz": mean_rate,
    "rate_std_hz": std_rate,
})
df.head()

# Plot with source curve overlay (if available)
plt.figure(figsize=(6, 4))
plt.plot(centers, mean_rate, label="mean")
plt.fill_between(centers, mean_rate - std_rate, mean_rate + std_rate, alpha=0.2, label="±std")
if ref_curve:
    ref_t, ref_r = ref_curve
    plt.plot(ref_t, ref_r, "--", color="orange", linewidth=1.5, label="source curve")
plt.xlabel("Time (ms)")
plt.ylabel("Rate (Hz per synapse)")
plt.title(f"{meta['group']}: mean of {meta['n_runs']} samples")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# --- Example: save a labeled single run --- 
do_save_example = False
if do_save_example:
    from modules import run_sim

    results = run_sim.run_sim(cell, geom, sim_cfg, groups_cfg, inputs_by_group)
    # tweak anything you like (color, meta, etc.)
    results["sim_cfg"]["color"] = "m"
    results["meta"]["note"] = "batch 1, bg_exc only"

    run_sim.save_results_with_name(results, "sst2_seg_tuned_bgexc_batch1")


In [ ]:
from pathlib import Path
from modules import run_sim, plotting

# load directly from disk; ignores current sim_cfg
do_load_results = False
if do_load_results:
    output_stem = "tune1"
    results_name = f"SST_seg_tuned_{output_stem}.pkl"
    base = Path("output_data")
    run_dir = base / output_stem
    results_file = run_dir / results_name
    legacy_file = base / results_name

    if run_dir.is_dir():
        results_path = run_dir
    elif results_file.is_file():
        results_path = results_file
    elif legacy_file.is_file():
        results_path = legacy_file
    else:
        raise FileNotFoundError(
            f"Set results_path to an existing run folder or file (got {run_dir})"
        )

    results = run_sim.load_results(results_path)

    # optional: tweak plot-only fields such as color
    results["sim_cfg"]["color"] = "m"

    # optional: build in_vivo_curve from current groups_cfg/sim_cfg (if you have them)
    # in_vivo_curve = (delayed_PFR_t, PFR_firing_rate_shortened)

    plotting.plot_results(results, in_vivo_curve=in_vivo_curve)


## Troubleshooting (Step 5)

If something breaks:
1. Bootstrap/setup failed: rerun first setup cell and verify repo path/token settings.
2. Mechanism errors: rerun modfile compile/load cell.
3. Input generation errors: check `cell_configs/syn_config.json` and `syn_groups/*.json`.
4. Empty/flat outputs: verify timing windows (`stim_start_ms`, duration, tstop).

Best practice for class labs:
- lock one baseline config,
- branch experiments from that baseline,
- change one variable group at a time.
